## Understanding Lasso Regression with ISTA

**Lasso (Least Absolute Shrinkage and Selection Operator) Regression** adds an **L1 penalty** to ordinary least squares. Unlike **Ridge (L2)** regression, Lasso can shrink coefficients **exactly to zero**, enabling **automatic feature selection**.

---

## Objective Function

$$
J(w,b) =
\underbrace{\frac{1}{2n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{MSE Loss (smooth)}}
+
\underbrace{\alpha \sum_{j=1}^{p} |w_j|}_{\text{L1 Penalty (non-smooth)}}
$$

---

## Why Not Regular Gradient Descent?

The L1 penalty $|w|$ is **not differentiable at zero**. While subgradient methods exist, they:

- Converge slowly  
- Do not naturally produce exact zeros  
- Are not the standard practical approach

---

## ISTA: The Standard Solution

The **Iterative Shrinkage-Thresholding Algorithm (ISTA)** handles the non-smooth L1 term by splitting optimization into two steps.

### Step 1: Gradient Descent on the Smooth MSE Part

$$
w_{\text{temp}} =
w - \eta \cdot \frac{1}{n} X^T (Xw + b - y)
$$

### Step 2: Proximal Operator (Soft-Thresholding)

$$
w_{\text{new}} = S(w_{\text{temp}}, \eta\alpha)
$$

---

## The Soft-Thresholding Operator

$$
S(w,\lambda) = \text{sign}(w) \cdot \max(|w| - \lambda, 0)
$$

Equivalent piecewise form:

$$
S(w,\lambda) =
\begin{cases}
w - \lambda & \text{if } w > \lambda \\
0 & \text{if } |w| \le \lambda \\
w + \lambda & \text{if } w < -\lambda
\end{cases}
$$

**Key Insight:**  
Values within $[-\lambda, \lambda]$ become **exactly zero**, which is how Lasso performs **feature selection**.

---

## Visual Intuition
```
Input w: -3 -2 -1 -0.5 0 0.5 1 2 3
λ = 1
Output: -2 -1 0 0 0 0 0 1 2
	All become exactly 0!
```

---

## Algorithm Summary

Initialize:$w = 0, b = 0$

For each iteration:

1. Predict: $ŷ = Xw + b$

2. Gradient step: $w_temp = w - η * (1/n) * X.T @ (ŷ - y)$

3. Soft-threshold:$w = sign(w_temp) * max(|w_temp| - ηα, 0)$

4. Update bias:$b = b - η * (1/n) * sum(ŷ - y)$

5. Check convergence

---

## Why ISTA Works

ISTA is a **proximal gradient method**.  
The soft-thresholding operator is the **proximal operator of the L1 norm**:

$$
\text{prox}_{\lambda \|\cdot\|_1}(w)
=
\arg\min_z
\left(
\frac{1}{2}\|z-w\|^2 + \lambda \|z\|_1
\right)
=
S(w,\lambda)
$$

This formulation has a **closed-form solution**, making ISTA:

- Efficient  
- Stable  
- Guaranteed to converge

[Problem Desc](https://www.deep-ml.com/problems/50?from=Deep%20Learning)

In [15]:
import numpy as np

def soft_threshold(w: np.ndarray, threshold: float) -> np.ndarray:
    """
    Apply soft-thresholding operator element-wise.
    
    S(w, λ) = sign(w) * max(|w| - λ, 0)
    
    Args:
        w: Input array
        threshold: Threshold value λ
    
    Returns:
        Soft-thresholded array where:
        - Values with |w| > λ are shrunk toward zero by λ
        - Values with |w| ≤ λ become exactly zero
    """
    return np.sign(w) * np.maximum(np.abs(w) - threshold, 0.0)

def l1_regularization_gradient_descent(
    X: np.ndarray, 
    y: np.ndarray, 
    alpha: float = 0.1, 
    learning_rate: float = 0.01, 
    max_iter: int = 1000, 
    tol: float = 1e-4) -> tuple:
    """
    Implement Lasso Regression using ISTA (Iterative Shrinkage-Thresholding Algorithm).
    
    ISTA alternates between:
    1. Gradient step on MSE loss: w_temp = w - lr * gradient_mse
    2. Proximal step (soft-thresholding): w_new = soft_threshold(w_temp, lr * alpha)
    
    Args:
        X: Feature matrix of shape (n_samples, n_features)
        y: Target vector of shape (n_samples,)
        alpha: L1 regularization strength
        learning_rate: Step size for gradient descent
        max_iter: Maximum iterations
        tol: Convergence tolerance on weight change
    
    Returns:
        tuple: (weights, bias)
    
    Note: The bias term is NOT regularized.
    """
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0.0
    
    for i in range(max_iter):
        pred = X @ weights + bias
        
        w_temp = weights - learning_rate * (1 / n_samples) * X.T @ (pred - y)
        
        w = soft_threshold(w_temp, learning_rate * alpha)
        
        bias = bias - learning_rate * (1/n_samples)* np.sum(pred - y)
        
        if np.max(np.abs(w - weights)) < tol:
            weights = w
            break
        
        weights = w
        
    return weights, bias

In [16]:
# Test Case 1
import numpy as np

# Test that Lasso produces sparse weights (some exactly zero)
X = np.array([[1, 0.01], [2, 0.02], [3, 0.03], [4, 0.04], [5, 0.05]])
y = np.array([2, 4, 6, 8, 10])  # y ≈ 2*X[:,0], X[:,1] is noise

weights, bias = l1_regularization_gradient_descent(X, y, alpha=0.5, learning_rate=0.01, max_iter=1000)
print(f"Sparse weight is zero: {weights[1] == 0}")

# Expected Output:
# Sparse weight is zero: True

Sparse weight is zero: True


In [17]:
# Test Case 2
import numpy as np

# Test that higher alpha produces more zeros
X = np.array([[1, 2, 0.1], [2, 4, 0.2], [3, 6, 0.3], [4, 8, 0.4]])
y = np.array([5, 10, 15, 20])

w_low, _ = l1_regularization_gradient_descent(X, y, alpha=0.01, learning_rate=0.01, max_iter=1000)
w_high, _ = l1_regularization_gradient_descent(X, y, alpha=1.0, learning_rate=0.01, max_iter=1000)

zeros_low = np.sum(w_low == 0)
zeros_high = np.sum(w_high == 0)
print(f"Higher alpha produces more zeros: {zeros_high >= zeros_low}")

# Expected Output:
# Higher alpha produces more zeros: True

Higher alpha produces more zeros: True


In [18]:
import numpy as np # Test basic functionality - should fit y = x well 
X = np.array([[1], [2], [3], [4], [5]]) 
y = np.array([1, 2, 3, 4, 5]) 
weights, bias = l1_regularization_gradient_descent(X, y, alpha=0.01, learning_rate=0.01, max_iter=1000) 
y_pred = X @ weights + bias 
mse = np.mean((y - y_pred) ** 2) 
print(f"MSE is small: {mse < 0.1}")

# Expected Output:
# MSE is small: True

MSE is small: True
